# Train (5-CV) and test using 93 datasets to find the best delta_max for each dataset (label for meta learning)

# Functions for Each Step

In [ ]:
import os
import pandas as pd
import numpy as np
import io
from scipy.io import arff
import tensorflow as tf
import math
import time

from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
from llmfilter_dh import describe_data, run_filter_chunked
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn import metrics
from sklearn.metrics import confusion_matrix
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, balanced_accuracy_score, precision_recall_curve, auc)
from imblearn.metrics import geometric_mean_score

from sklearn.model_selection import ParameterGrid
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm, neighbors
from lightgbm import LGBMClassifier
from sklearn.base import clone

import re
############################ Data feature names cleaning ############################
# This function cleans 'thoughts ?' -> 'thoughts?' and handles extra spaces
def clean_column_names(df):
    new_columns = []
    for col in df.columns:
        # 1. Remove leading/trailing whitespace
        c = col.strip()
        # 2. Replace multiple spaces with a single space
        c = re.sub(r'\s+', ' ', c)
        # 3. Remove space before punctuation (e.g., " ?" -> "?")
        c = re.sub(r'\s+([?.!,])', r'\1', c)
        new_columns.append(c)
    
    df.columns = new_columns
    return df

############################ Step 1: Identify Feature Types and Numericalize ############################
def identify_feature_types(df, t=15):
    metadata_list = []
    
    for col in df.columns:
        unique_vals = np.sort(df[col].unique())
        n_unique = len(unique_vals)
        dtype = df[col].dtype
        
        # --- Logic Gates for Categorization ---
        if n_unique <= t:
            # All low-cardinality features are treated as Categorical (Snapping required)
            category = 'Categ'
        
        else:
            if np.issubdtype(dtype, np.integer):
                # High-cardinality integers (Age, Study Hours)
                category = 'Num_disc'
            elif dtype == 'object':
                # High-cardinality strings (Names, IDs, Addresses)
                category = 'REMOVE'
            else:
                category = 'Num_cont' # Fallback for unknown numeric types
        
        metadata_list.append({
            'Feature': col,
            'Num Unique': n_unique,
            'Data Type': dtype,
            'Valid Values': unique_vals.tolist(),
            'Category': category
        })
    metadata = pd.DataFrame(metadata_list)
    Categ = [metadata.loc[i, 'Feature'] for i in range(len(metadata)) if metadata.loc[i, 'Category'] == 'Categ']
    Num_disc = [metadata.loc[i, 'Feature'] for i in range(len(metadata)) if metadata.loc[i, 'Category'] == 'Num_disc']
    Num_cont = [metadata.loc[i, 'Feature'] for i in range(len(metadata)) if metadata.loc[i, 'Category'] == 'Num_cont']
    REMOVE = [metadata.loc[i, 'Feature'] for i in range(len(metadata)) if metadata.loc[i, 'Category'] == 'REMOVE']
        
    return metadata, Categ, Num_disc, Num_cont, REMOVE

def encode_to_numerical(df_org, REMOVE, Categ):
    # 1. Remove features in the REMOVE list
    df_cleaned = df_org.drop(REMOVE, axis=1)

    # 2. Initialize the translation dictionary (The "Rosetta Stone")
    # We will need this for Step 5 and Step 7
    mapping_dict = {}

    df_num = df_cleaned.copy()

    for col in Categ:
        # Get the valid values we stored in metadata
        # We sort them to ensure the mapping is consistent (e.g., 0 is always the same thing)
        unique_labels = sorted(df_num[col].unique())
        if col == Categ[-1]:
            # label = last column (always, Major = 0, Minor = 1)
            unique_labels = [df_num.iloc[:,-1].value_counts().index[0], df_num.iloc[:,-1].value_counts().index[1]]

        # Create the Forward (Text -> Num) and Reverse (Num -> Text) maps
        forward_map = {label: i for i, label in enumerate(unique_labels)}
        reverse_map = {i: label for i, label in enumerate(unique_labels)}

        # Save to our dictionary
        mapping_dict[col] = {
            'forward': forward_map,
            'reverse': reverse_map
        }

        # Apply the transformation
        df_num[col] = df_num[col].map(forward_map)

    # 3. Ensure all columns are numeric types (float or int) for SMOTE
    # This converts any remaining "object" types that might have slipped through
    df_num = df_num.apply(pd.to_numeric)
    
    return mapping_dict, df_num

############################ Step 2: Extra SMOTE generates new data points with Extrapolation ############################
def smote_generate(data, n_samples, delta_range=(0.0, 1.0), seed=None):
    # Set the seed for reproducibility if a number is provided
    # If seed is None, it uses the system clock (different every time)
    smote_k = 5                                                      # for knn in smote or extra smote
    rng = np.random.default_rng(seed)
    knn = NearestNeighbors(n_neighbors=smote_k).fit(data)            # Find k nearest neighbors for each minority point
    synthetic = []
    for _ in range(n_samples):
        idx = rng.integers(0, len(data))                             # Pick a random index from minority points
        origin = data[idx]                                           # the chosen minority point
        neighbor_distances, neighbor_indices = knn.kneighbors([origin])
        neighbor = data[rng.choice(neighbor_indices[0][1:])]         # Pick a random neighbor (excluding itself)  
        delta = rng.uniform(delta_range[0], delta_range[1])          # The random factor
        synthetic.append(origin + delta * (neighbor - origin))       # Formula: x_new = x_i + delta * (x_neighbor - x_i)
    return pd.DataFrame(synthetic, columns = df.columns[:-1])

############################ Step 3: Snap & Round (correction for original-like data) ############################
def process_extra_samples(df_raw_extra, Num_cont, Num_disc, Categ, meta_df):
    """
    Snaps and rounds raw synthetic data based on feature classifications.
    Input df_raw_extra is a DataFrame.
    """
    df_extra = df_raw_extra.copy()
    
    # FIX: Ensure 'Feature' is the index so we can look up by name
    # We check if 'Feature' is a column; if so, we set it as index.
    if 'Feature' in meta_df.columns:
        meta_df = meta_df.set_index('Feature')
    
    for col in df_extra.columns:
        if col in Num_cont:
            continue
            
        elif col in Num_disc:
            df_extra[col] = df_extra[col].round().astype(int)
            
        elif col in Categ:
            # Look up the number of categories
            # Note: Ensure this matches the column name in your metadata ('Num Unique')
            num_categories = meta_df.loc[col, 'Num Unique']
            
            # Valid options are the integer indices 0, 1, ..., n-1
            valid_indices = np.arange(num_categories)
            
            def snap_to_closest(val):
                # Using absolute difference to find the nearest integer label
                idx = (np.abs(valid_indices - val)).argmin()
                return valid_indices[idx]
            
            df_extra[col] = df_extra[col].apply(snap_to_closest).astype(int)

    return df_extra

############################ Step 4: ENN filtering ############################
def enn_filter(synthetic_points, orig_min, orig_maj):
    enn_k = 3                                                 # for knn in enn
    X_ref = np.vstack([orig_min, orig_maj])                   # Original whole data
    y_ref = np.array([1]*len(orig_min) + [0]*len(orig_maj))   # Original data's label
    knn = NearestNeighbors(n_neighbors=enn_k).fit(X_ref)     
    
    kept, removed = [], []
    for p in synthetic_points:                                # For each extra-generated point
        distances, indices = knn.kneighbors(p.reshape(1, -1)) # Find its neighbors' indices
        # If majority of neighbors in original data are minority class, KEEP.
        if np.sum(y_ref[indices[0]]) > (enn_k / 2):
            kept.append(p)
        else:
            removed.append(p)
    return pd.DataFrame(kept, columns = df.columns[:-1]), pd.DataFrame(removed, columns = df.columns[:-1])

In [ ]:
def get_results(y, predicted, pred_prob):
    # Initialize all variables to zero to prevent UnboundLocalError
    acc = pre = rec = spe = f1 = gmean = bacc = rauc = prauc = 0.0
    
    # Check for NaNs or empty predictions
    if np.isnan(predicted).any() or len(predicted) == 0:
        return acc, pre, rec, spe, f1, gmean, bacc, rauc, prauc
    
    else:
        # 1. Efficient Confusion Matrix extraction
        # ravel() returns tn, fp, fn, tp in one line for binary classification
        try:
            tn, fp, fn, tp = metrics.confusion_matrix(y, predicted, labels=[0, 1]).ravel()
        except ValueError: 
            # Handles cases where only one class is present in y or predicted
            tn = tp = fp = fn = 0 

        # 2. Standard Metrics
        acc = np.round(accuracy_score(y, predicted), 4)
        pre = np.round(precision_score(y, predicted, zero_division=0), 4)
        rec = np.round(recall_score(y, predicted, zero_division=0), 4)
        
        # Specificity: TN / (TN + FP)
        spe = np.round(tn / (tn + fp), 4) if (tn + fp) > 0 else 0.0
        
        f1 = np.round(f1_score(y, predicted, zero_division=0), 4)
        gmean = np.round(geometric_mean_score(y, predicted), 4)
        bacc = np.round(balanced_accuracy_score(y, predicted), 4)
        
        # 3. Probability-based Metrics (AUC)
        try:
            rauc = np.round(roc_auc_score(y, pred_prob), 4)
        except:
            rauc = 0.0
            
        try:
            precision, recall, _ = precision_recall_curve(y, pred_prob)
            prauc = np.round(auc(recall, precision), 4)
        except:
            prauc = 0.0
    
    return acc, pre, rec, spe, f1, gmean, bacc, rauc, prauc

In [ ]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state = 2)   # 5-fold-cross validation

## Validation (70 => train:val = 56:14, 5CV) and Test(30)

In [ ]:
# 1. Define the Optimized Parameter Grids
configs = [
    # Logistic Regression (7) - Linear Classifier
    (LogisticRegression, {
        'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000], 
        'max_iter': [100000], 'random_state': [100]
    }),
               
    # LGBM (48) - Boosting-based Classifier
    (LGBMClassifier, {
        'boosting_type': ['gbdt', 'dart', 'goss'], 
        'max_depth': [10, 20], 
        'learning_rate': [0.01, 0.05], 
        'n_estimators': [50, 100, 150, 200], 
        'random_state': [100], 'verbose': [-1]
    })
]

# 2. Build Models and ModelNames in Parallel
Model = []
ModelName = []

config_map = {
    LogisticRegression: 'LR',
    DecisionTreeClassifier: 'DT',
    RandomForestClassifier: 'RF',
    svm.SVC: 'SVM',
    neighbors.KNeighborsClassifier: 'KNN',
    LGBMClassifier: 'LG'
}

for clf_class, params in configs:
    prefix = config_map[clf_class]
    grid = list(ParameterGrid(params))
    for i, p in enumerate(grid):
        Model.append(clf_class(**p))
        ModelName.append(f"{prefix}{i+1}")

print(f"Total Models: {len(Model)}") 
print(ModelName)

In [ ]:
import warnings
warnings.filterwarnings(action='ignore')

# Experiments (XE {delta=1.5/2/2.5/3})

In [ ]:
for i in range(1, 98):
    start = time.time()
    if i == 23 or i == 81 or i == 82 or i == 84:        # duplicated
        continue
    # Load data
    df = pd.read_csv(r'/5. SMOTE_LLM/data_original/'+'ds'+ str(i) +'.csv')
    df = clean_column_names(df)
    # STEP1: Feature Types & Numericalize
    metadata, Categ, Num_disc, Num_cont, REMOVE = identify_feature_types(df, t=15)
    mapping_dict, df = encode_to_numerical(df, REMOVE, Categ)  
    print('+'*35, '{}th Dataset'.format(i), '+'*35)
    print('<Whole-Data Class Dist>\n', df.iloc[:,-1].value_counts())
    print('<Whole-Data IMB ratio>\n', "1:{: .4f}".format(df.iloc[:,-1].value_counts()[1]/df.iloc[:,-1].value_counts()[0]))
   
    ##################### Validation:Test = 70:30 #######################
    df_val, df_test = train_test_split(df, test_size=0.3, random_state=100, stratify=df.iloc[:,-1])
    X = df_val.iloc[:, :-1]        # For validation (70%)
    y = df_val.iloc[:, -1]         # For validation (70%)
    X_test = df_test.iloc[:, :-1]  # For test (30%)
    y_test = df_test.iloc[:, -1]   # For test (30%)
    X_test = np.array(X_test).astype(float)
    y_test = np.array(y_test).astype(float)
    
    res_df = pd.DataFrame({'Dataset': [str(i) for a in range(27)]},
                   index = ['Acc_tr','Pre_tr','Rec_tr','Spe_tr','F1_tr','Gmean_tr','B_Acc_tr','R-AUC_tr','PR-AUC_tr',
                            'Acc_val','Pre_val','Rec_val','Spe_val','F1_val','Gmean_val','B_Acc_val','R-AUC_val','PR-AUC_val',
                            'Acc_t','Pre_t','Rec_t','Spe_t','F1_t','Gmean_t','B_Acc_t','R-AUC_t','PR-AUC_t'])
    Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]  
    ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
    min_strategy = Strategy[ind]
    if i == 33:
        min_strategy = Strategy[1] 
    print("<min_strategy>:",min_strategy)     
    
    ##################### With Data Generation #######################
    delta_values = [1.5, 2, 2.5, 3]
    for delta in delta_values:
        for j in range(len(Strategy)):
            print("==========", "delta:{}_{}".format(delta, Strategy[j]), "==========") 
            for k in range(len(Model)): 
                tr_acc = []
                tr_pre = []
                tr_rec = []
                tr_spe = []
                tr_f1 = []
                tr_gmean = []
                tr_bacc = []
                tr_rauc = []
                tr_prauc = []

                val_acc = []
                val_pre = []
                val_rec = []
                val_spe = []
                val_f1 = []
                val_gmean = []
                val_bacc = []
                val_rauc = []
                val_prauc = []

                if min_strategy > Strategy[j]:
                    res_df['XE(d:{})_{}_{}'.format(delta, Strategy[j], ModelName[k])] = [0 for c in range(27)]

                else:               
                    # 5-fold-CV
                    n_iter=0
                    for train_index, val_index in skf.split(X, y):
                        model = clone(Model[k])         # call the clean model
                        n_iter += 1
                        X_train = X.iloc[train_index]
                        y_train= y.iloc[train_index]
                        if k == 0 and n_iter == 1:
                            print("CV_TRAIN(0/1/total):", list(y_train).count(0), list(y_train).count(1), len(y_train))     
                        # Loading Generated Data
                        over_df = pd.read_csv(r'./Generated Data/XE/(delta='+str(delta)+')ds'
                                          +str(i)+'_XE_'+str(Strategy[j])+'_'+str(n_iter)+'th.csv')
                        X_train = over_df.iloc[:, :-1]
                        y_train = over_df.iloc[:, -1]     
                        if k == 0 and n_iter == 1:
                            print("CV_TRAIN_over(0/1/total):", list(y_train).count(0), list(y_train).count(1), len(y_train)) 
                            
                        X_val = X.iloc[val_index]
                        y_val= y.iloc[val_index]
                        if k == 0 and n_iter == 1:
                            print("CV_VALIDATION(0/1/total):", list(y_val).count(0), list(y_val).count(1), len(y_val))
                        # Array
                        X_train = np.array(X_train).astype(float)
                        y_train = np.array(y_train).astype(float)
                        X_val = np.array(X_val).astype(float)
                        y_val = np.array(y_val).astype(float)
                        # Learning
                        model.fit(X_train, y_train)
                        train_result = model.predict(X_train)
                        train_result_prob = model.predict_proba(X_train)[:, 1] # Get probability of class 1
                        acc, pre, rec, spe, f1, gmean, bacc, rauc, prauc = get_results(y_train, train_result, train_result_prob)
                        tr_acc.append(acc)
                        tr_pre.append(pre)
                        tr_rec.append(rec)
                        tr_spe.append(spe)
                        tr_f1.append(f1)
                        tr_gmean.append(gmean)
                        tr_bacc.append(bacc)
                        tr_rauc.append(rauc) 
                        tr_prauc.append(prauc) 
                        # Results 
                        result = model.predict(X_val)
                        result_prob = model.predict_proba(X_val)[:, 1] # Get probability of class 1
                        acc, pre, rec, spe, f1, gmean, bacc, rauc, prauc = get_results(y_val, result, result_prob)
                        val_acc.append(acc)
                        val_pre.append(pre)
                        val_rec.append(rec)
                        val_spe.append(spe)
                        val_f1.append(f1)
                        val_gmean.append(gmean)
                        val_bacc.append(bacc)
                        val_rauc.append(rauc)
                        val_prauc.append(prauc)

                    # Test
                    model = clone(Model[k])         # call the clean model
                    over_df = pd.read_csv(r'./Generated Data/XE/(delta='+str(delta)+')ds'
                                          +str(i)+'_XE_'+str(Strategy[j])+'_full.csv')
                    X_over = over_df.iloc[:, :-1]
                    y_over = over_df.iloc[:, -1]
                    if k == 0:
                        print("TRAIN_over(0/1/total):", list(y_over).count(0), list(y_over).count(1), len(y_over))
                        print("TEST(0/1/total):", list(y_test).count(0), list(y_test).count(1), len(y_test))
                    model.fit(np.array(X_over).astype(float), np.array(y_over).astype(float))
                    result = model.predict(X_test)
                    result_prob = model.predict_proba(X_test)[:, 1] # Get probability of class 1
                    acc_t, pre_t, rec_t, spe_t, f1_t, gmean_t, bacc_t, rauc_t, prauc_t = get_results(y_test, result, result_prob)

                    res_df['XE(d:{})_{}_{}'.format(delta, Strategy[j], ModelName[k])] = [
                        np.mean(tr_acc),np.mean(tr_pre),np.mean(tr_rec),np.mean(tr_spe),np.mean(tr_f1),
                        np.mean(tr_gmean),np.mean(tr_bacc),np.mean(tr_rauc),np.mean(tr_prauc),
                        np.mean(val_acc),np.mean(val_pre),np.mean(val_rec),np.mean(val_spe),np.mean(val_f1),
                        np.mean(val_gmean),np.mean(val_bacc),np.mean(val_rauc),np.mean(val_prauc),
                        acc_t, pre_t, rec_t, spe_t, f1_t, gmean_t, bacc_t, rauc_t, prauc_t]
            
    end = time.time()
    print(end-start)
    res_df.to_csv('./Meta Results/XE(deltas)_result.csv', mode = 'a', float_format='%.4g')